# Reference finder (Crossref) - run this one cell, then verify
Searches Crossref for each topic and shows the top 5 matches **with the error message if a search fails** (the previous version skipped silently). Download `reference_candidates_VERIFY.csv`, open every DOI, and keep only the correct paper. Hansen's handbook and TOPSIS (Hwang & Yoon 1981) are books: cite them manually.

In [ ]:
import requests, pandas as pd, time
from google.colab import files
EMAIL = "your.email@student.sust.edu"   # optional: Crossref gives faster service when an email is given
REFQ = {
 "Cyrene as dipolar aprotic solvent (Sherwood 2014)": "Dihydrolevoglucosenone (Cyrene) as a bio-based alternative for dipolar aprotic solvents",
 "Kamlet-Taft parameters of solvents (Jessop 2012)": "Jessop Kamlet-Taft parameters solvents green chemistry",
 "CHEM21 solvent selection guide": "CHEM21 selection guide of classical- and less classical-solvents",
 "GSK solvent selection guide": "Expanding GSK's solvent selection guide",
 "GVL sustainable liquid (Horvath 2008)": "gamma-Valerolactone a sustainable liquid for energy and carbon-based chemicals",
 "Butina clustering": "Unsupervised data base clustering based on Daylight's fingerprint and Tanimoto similarity",
 "SHAP (Lundberg and Lee)": "A unified approach to interpreting model predictions",
 "scikit-learn": "Scikit-learn: Machine learning in Python",
 "Replacement of DMF/NMP review": "alternatives to dipolar aprotic solvents DMF NMP DMAc replacement review",
 "Bio-based solvents Hansen parameters": "Hansen solubility parameters bio-based solvents",
 "QSPR solvent property machine learning": "machine learning prediction Kamlet-Taft solvent parameters",
 "OECD QSAR validation principles": "OECD principles for the validation of QSAR models",
}
rows = []
for topic, q in REFQ.items():
    try:
        r = requests.get("https://api.crossref.org/works", params={"query.bibliographic": q, "rows": 5, "mailto": EMAIL}, timeout=40)
        r.raise_for_status()
        for k, it in enumerate(r.json()["message"]["items"]):
            au = "; ".join(f"{a.get('family','')} {a.get('given','')[:1]}" for a in (it.get("author") or [])[:4])
            rows.append(dict(topic=topic, rank=k+1, title=(it.get("title") or [""])[0], authors=au, journal=(it.get("container-title") or [""])[0],
                             year=((it.get("issued") or {}).get("date-parts") or [[None]])[0][0], volume=it.get("volume",""), pages=it.get("page",""), doi=it.get("DOI","")))
    except Exception as e:
        print("FAILED:", topic, "->", repr(e)[:160])
    time.sleep(0.5)
REF = pd.DataFrame(rows); pd.set_option("display.max_colwidth", 80, "display.width", 250)
display(REF[["topic","rank","title","authors","year","doi"]])
REF.to_csv("reference_candidates_VERIFY.csv", index=False); files.download("reference_candidates_VERIFY.csv")